In [ ]:
   # // AgriVision - Copyright (C) 2026 <Shahd Kazan>
   # // Licensed under the GNU AGPL-3.0. See the LICENSE file.

In [1]:
# =============================================================
# 1. Install dependencies
# =============================================================
!pip install -q ultralytics opencv-python-headless

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 6.7 MB/s eta 0:00:00


In [ ]:
# =============================================================
!pip install -q ultralytics tensorflow opencv-python-headless scikit-learn pandas

# =============================================================
# 1. IMPORTS
# =============================================================
import os
import gc
import cv2
import yaml
import shutil
import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from pathlib import Path
from ultralytics import YOLO
from sklearn.model_selection import StratifiedShuffleSplit

# =============================================================
# 2. PATHS
# =============================================================
DATA_ROOT = Path(
    "/kaggle/input/datasets/social888/plantdisease/leaf disease.yolov8"
)

WORKDIR = Path("/kaggle/working")

DATASET_DIR = WORKDIR / "dataset"

DATASET_DIR.mkdir(parents=True, exist_ok=True)

PT_SRC = (
    "path/to/evaluated/model.pt"
)

PT_WORK = WORKDIR / "model.pt"

# =============================================================
# 3. COPY MODEL TO WRITABLE SPACE
# =============================================================
shutil.copy(PT_SRC, PT_WORK)

# =============================================================
# 4. LOAD METADATA
# =============================================================
meta = pd.read_csv(DATA_ROOT / "metadata.csv")

meta["strata"] = (
    meta["crop"].astype(str)
    + "_"
    + meta["label"].astype(str)
    + "_"
    + meta["environment"].astype(str)
)

# =============================================================
# 5. STRATIFIED SPLIT
# =============================================================
split1 = StratifiedShuffleSplit(
    n_splits=1,
    test_size=0.1,
    random_state=42
)

train_val_idx, test_idx = next(
    split1.split(meta, meta["strata"])
)

train_val = meta.iloc[train_val_idx]
test_df = meta.iloc[test_idx]

split2 = StratifiedShuffleSplit(
    n_splits=1,
    test_size=0.11,
    random_state=42
)

train_idx, val_idx = next(
    split2.split(train_val, train_val["strata"])
)

train_df = train_val.iloc[train_idx]
val_df = train_val.iloc[val_idx]

# =============================================================
# 6. BUILD YOLO DATASET
# =============================================================
def copy_split(df, name):

    out = DATASET_DIR / name

    (out / "images").mkdir(parents=True, exist_ok=True)
    (out / "labels").mkdir(parents=True, exist_ok=True)

    for img in df["image_name"]:

        shutil.copy(
            DATA_ROOT / "train/images" / img,
            out / "images" / img
        )

        label = img.replace(".jpg", ".txt")

        src_label = DATA_ROOT / "train/labels" / label

        if src_label.exists():

            shutil.copy(
                src_label,
                out / "labels" / label
            )

copy_split(train_df, "train")
copy_split(val_df, "val")
copy_split(test_df, "test")

# =============================================================
# 7. DATA YAML
# =============================================================
yaml_path = WORKDIR / "data.yaml"

yaml_path.write_text(f"""
path: {DATASET_DIR}

train: train/images
val: val/images
test: test/images

nc: 9

names:
  0: maize fall armyworm
  1: maize leaf
  2: maize leaf blight
  3: potato late blight
  4: potato leaf
  5: potato pest damage
  6: tomato early blight
  7: tomato leaf
  8: tomato leaf miner
""")


# =============================================================
# 8. LOAD PT MODEL
# =============================================================
pt_model = YOLO(str(PT_WORK))

# =============================================================
# 9. CREATE CALIBRATION DATASET FOR INT8
# =============================================================
# Ultralytics uses the dataset YAML for representative calibration.

print("\nCalibration dataset ready.")

# =============================================================
# 10. EXPORT FLOAT32 TFLITE
# =============================================================
print("\nExporting FLOAT32 TFLite...")

pt_model.export(
    format="tflite",
    imgsz=640,
    half=False,
    int8=False
)

# =============================================================
# 11. EXPORT FLOAT16 TFLITE
# =============================================================
print("\nExporting FLOAT16 TFLite...")

pt_model.export(
    format="tflite",
    imgsz=640,
    half=True,
    int8=False
)

# =============================================================
# 12. EXPORT INT8 QUANTIZED TFLITE
# =============================================================
print("\nExporting INT8 Quantized TFLite...")

pt_model.export(
    format="tflite",
    imgsz=640,
    int8=True,
    data=str(yaml_path)   # calibration dataset
)

# =============================================================
# 13. FIND EXPORTED MODELS
# =============================================================
all_tflite_files = list(
    Path("/kaggle/working").rglob("*.tflite")
)

if len(all_tflite_files) == 0:
    raise RuntimeError("No TFLite files found.")

print("\nFound TFLite Models:\n")

for f in all_tflite_files:
    print(f)

float32_path = None
float16_path = None
int8_path = None

for f in all_tflite_files:

    f_str = str(f).lower()

    # FLOAT16
    if "float16" in f_str:
        float16_path = f

    # INT8
    elif "int8" in f_str:
        int8_path = f

    # FLOAT32
    elif "float32" in f_str:
        float32_path = f

# fallback protections
if float32_path is None:
    for f in all_tflite_files:
        if "float16" not in str(f).lower() and "int8" not in str(f).lower():
            float32_path = f
            break

if float16_path is None:
    raise RuntimeError("FLOAT16 model not found.")

if int8_path is None:
    raise RuntimeError("INT8 model not found.")

if float32_path is None:
    raise RuntimeError("FLOAT32 model not found.")

print("\nUsing FLOAT32:")
print(float32_path)

print("\nUsing FLOAT16:")
print(float16_path)

print("\nUsing INT8:")
print(int8_path)

# =============================================================
# 14. LOAD MODELS
# =============================================================
pt_eval_model = YOLO(
    str(PT_WORK),
    task="segment"
)

tflite32_model = YOLO(
    str(float32_path),
    task="segment"
)

tflite16_model = YOLO(
    str(float16_path),
    task="segment"
)

tflite_int8_model = YOLO(
    str(int8_path),
    task="segment"
)

# =============================================================
# 15. EVALUATION FUNCTION
# =============================================================
def evaluate_model(model, name):

    print("\n")
    print("=" * 60)
    print(f"EVALUATING: {name}")
    print("=" * 60)

    metrics = model.val(
        data=str(yaml_path),
        split="test",
        imgsz=640,
        conf=0.25,
        verbose=False
    )

    results = {

        "Model": name,

        # BOX METRICS
        "Box_mAP50": metrics.box.map50,
        "Box_mAP50_95": metrics.box.map,
        "Box_Precision": metrics.box.mp,
        "Box_Recall": metrics.box.mr,

        # SEG METRICS
        "Seg_mAP50": metrics.seg.map50,
        "Seg_mAP50_95": metrics.seg.map,
        "Seg_Precision": metrics.seg.mp,
        "Seg_Recall": metrics.seg.mr,
    }

    for k, v in results.items():

        if isinstance(v, float):
            print(f"{k}: {v:.4f}")
        else:
            print(f"{k}: {v}")

    return results

# =============================================================
# 16. RUN ALL EVALUATIONS
# =============================================================
pt_results = evaluate_model(
    pt_eval_model,
    "PyTorch (.pt)"
)

float32_results = evaluate_model(
    tflite32_model,
    "TFLite FLOAT32"
)

float16_results = evaluate_model(
    tflite16_model,
    "TFLite FLOAT16"
)

int8_results = evaluate_model(
    tflite_int8_model,
    "TFLite INT8"
)

# =============================================================
# 17. FINAL COMPARISON TABLE
# =============================================================
comparison_df = pd.DataFrame([

    pt_results,
    float32_results,
    float16_results,
    int8_results

])

print("\n")
print("=" * 80)
print("FINAL MODEL COMPARISON small-26-640")
print("=" * 80)

display(comparison_df)

# =============================================================
# 18. DEPLOYMENT BENCHMARK METRICS
# =============================================================
import time
import os

# =============================================================
# BENCHMARK FUNCTION
# =============================================================
def benchmark_model(
    model,
    model_name,
    model_path,
    sample_image,
    warmup_runs=5,
    benchmark_runs=20,
    pt_model_for_params=None
):

    print("\n")
    print("=" * 60)
    print(f"BENCHMARKING: {model_name}")
    print("=" * 60)

    # ---------------------------------------------------------
    # WARMUP
    # ---------------------------------------------------------
    print("Running warmup...")

    for _ in range(warmup_runs):

        _ = model.predict(
            source=str(sample_image),
            imgsz=640,
            conf=0.25,
            verbose=False
        )

    # ---------------------------------------------------------
    # TIMED RUNS
    # ---------------------------------------------------------
    print("Running benchmark...")

    latencies = []

    for _ in range(benchmark_runs):

        start_time = time.perf_counter()

        _ = model.predict(
            source=str(sample_image),
            imgsz=640,
            conf=0.25,
            verbose=False
        )

        latency_ms = (
            time.perf_counter() - start_time
        ) * 1000

        latencies.append(latency_ms)

    # ---------------------------------------------------------
    # METRICS
    # ---------------------------------------------------------
    avg_ms = np.mean(latencies)

    fps = 1000.0 / avg_ms

    latency_std = np.std(latencies)

    size_mb = (
        os.path.getsize(model_path)
        / (1024 * 1024)
    )

    # ---------------------------------------------------------
    # PARAM COUNT
    # ---------------------------------------------------------
    params_M = None

    if pt_model_for_params is not None:

        total_params = sum(
            p.numel()
            for p in pt_model_for_params.model.parameters()
        )

        params_M = total_params / 1e6

    results = {

        "Model": model_name,

        "avg_ms": avg_ms,
        "fps": fps,
        "latency_std": latency_std,

        "size_mb": size_mb,

        "params_M": params_M,
    }

    for k, v in results.items():

        if isinstance(v, float):
            print(f"{k}: {v:.4f}")
        else:
            print(f"{k}: {v}")

    return results
test_images = list(
    (DATASET_DIR / "test/images").glob("*.*")
)

print("\nTotal Test Images:", len(test_images))

# =============================================================
# SAMPLE IMAGE FOR BENCHMARKING
# =============================================================
benchmark_image = random.choice(test_images)

print("\nBenchmark image:")
print(benchmark_image)

# =============================================================
# RUN BENCHMARKS
# =============================================================
pt_benchmark = benchmark_model(
    model=pt_eval_model,
    model_name="PyTorch (.pt)",
    model_path=str(PT_WORK),
    sample_image=benchmark_image,
    pt_model_for_params=pt_eval_model
)

float32_benchmark = benchmark_model(
    model=tflite32_model,
    model_name="TFLite FLOAT32",
    model_path=str(float32_path),
    sample_image=benchmark_image,
    pt_model_for_params=pt_eval_model
)

float16_benchmark = benchmark_model(
    model=tflite16_model,
    model_name="TFLite FLOAT16",
    model_path=str(float16_path),
    sample_image=benchmark_image,
    pt_model_for_params=pt_eval_model
)

int8_benchmark = benchmark_model(
    model=tflite_int8_model,
    model_name="TFLite INT8",
    model_path=str(int8_path),
    sample_image=benchmark_image,
    pt_model_for_params=pt_eval_model
)

# =============================================================
# Benchmark pt
# =============================================================
# fp32_size = float32_benchmark["size_mb"]

# fp32_latency = float32_benchmark["avg_ms"]
pt_size = pt_benchmark["size_mb"]

pt_latency = pt_benchmark["avg_ms"]
# =============================================================
# COMPUTE COMPRESSION + SPEEDUP
# =============================================================
all_benchmarks = [

    pt_benchmark,
    float32_benchmark,
    float16_benchmark,
    int8_benchmark

]

for result in all_benchmarks:

    # result["compression_ratio"] = (
    #     result["size_mb"] / fp32_size
    # )

    # result["speedup_vs_fp32"] = (
    #     fp32_latency / result["avg_ms"]
    # )

    result["compression_ratio_vs_pt"] = (
        result["size_mb"] / pt_size
    )

    result["speedup_vs_pt"] = (
        pt_latency / result["avg_ms"]
    )
# =============================================================
# DEPLOYMENT RESULTS TABLE
# =============================================================
deployment_df = pd.DataFrame(all_benchmarks)

print("\n")
print("=" * 80)
print("DEPLOYMENT METRICS")
print("=" * 80)

display(deployment_df)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.

Calibration dataset ready.

Exporting FLOAT32 TFLite...
Ultralytics 8.4.57 🚀 Python-3.12.12 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLO26s-seg summary (fused): 139 layers, 10,368,823 parameters, 0 gradients, 34.1 GFLOPs

PyTorch: starting from '/kaggle/working/model.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) ((1, 300, 38), (1, 32, 160, 160)) (22.3 MB)
requirements: Ultralytics requirements ['onnxslim>=0.1.82', 'onnxruntime'] not found, attempting AutoUpdate...
Using Python 3.12.12 environment at: /usr
Resolved 1

/usr/local/lib/python3.12/dist-packages/torch/onnx/_internal/torchscript_exporter/symbolic_opset11.py:954: UserWarning: Exporting aten::index operator of advanced indexing in opset 20 is achieved by combination of multiple ONNX operators, including Reshape, Transpose, Concat, and Gather. If indices include negative values, the exported graph will produce incorrect results.
  return opset9.index(g, self, index)


ONNX: slimming with onnxslim 0.1.94...
ONNX: export success ✅ 5.8s, saved as '/kaggle/working/model.onnx' (40.0 MB)


E0000 00:00:1780168055.124231      16 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780168055.193654      16 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780168055.731233      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780168055.731304      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780168055.731307      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780168055.731310      16 computation_placer.cc:177] computation placer already registered. Please check linka

requirements: Ultralytics requirement ['onnx2tf>=1.26.3,<1.29.0'] not found, attempting AutoUpdate...
Using Python 3.12.12 environment at: /usr
Resolved 1 package in 232ms
Prepared 1 package in 290ms
Installed 1 package in 7ms
 + onnx2tf==1.28.8

requirements: AutoUpdate success ✅ 0.7s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

requirements: Ultralytics requirements ['sng4onnx>=1.0.1', 'onnx_graphsurgeon>=0.3.26', 'ai-edge-litert>=1.2.0'] not found, attempting AutoUpdate...
Using Python 3.12.12 environment at: /usr
Resolved 11 packages in 3.07s
 Downloaded ai-edge-litert
Prepared 4 packages in 759ms
Installed 4 packages in 7ms
 + ai-edge-litert==2.1.5
 + backports-strenum==1.3.1
 + onnx-graphsurgeon==0.6.1
 + sng4onnx==2.0.1

requirements: AutoUpdate success ✅ 4.0s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


TensorFlow SavedModel: starting export with tensorflow 2.19.0...
Unzipping calibration_image_s

I0000 00:00:1780168117.733092      16 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 0
I0000 00:00:1780168117.733381      16 single_machine.cc:374] Starting new session
W0000 00:00:1780168124.943990      16 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1780168124.944080      16 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
I0000 00:00:1780168129.320944      16 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 0
I0000 00:00:1780168129.321196      16 single_machine.cc:374] Starting new session
W0000 00:00:1780168134.910104      16 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1780168134.910193      16 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.


... (7 outputs left out)

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 202/202 1.4it/s 2:23
                   all        202        852       0.69      0.594      0.536      0.394      0.693      0.588      0.535      0.371
Speed: 1.3ms preprocess, 649.7ms inference, 0.0ms loss, 1.2ms postprocess per image
Results saved to /kaggle/working/runs/segment/val-2
Model: TFLite FLOAT32
Box_mAP50: 0.5357
Box_mAP50_95: 0.3937
Box_Precision: 0.6902
Box_Recall: 0.5943
Seg_mAP50: 0.5345
Seg_mAP50_95: 0.3706
Seg_Precision: 0.6930
Seg_Recall: 0.5883


EVALUATING: TFLite FLOAT16
Ultralytics 8.4.57 🚀 Python-3.12.12 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
Loading /kaggle/working/model_saved_model/model_float16.tflite for TensorFlow Lite inference...
Setting batch=1 input of shape (1, 3, 640, 640)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1851.3±1056.0 MB/s, size: 1855.1 KB)
val: Scanning /kaggle/working/da

,Model,Box_mAP50,Box_mAP50_95,Box_Precision,Box_Recall,Seg_mAP50,Seg_mAP50_95,Seg_Precision,Seg_Recall
0,PyTorch (.pt),0.558084,0.410351,0.676166,0.615187,0.564406,0.387863,0.678007,0.615799
1,TFLite FLOAT32,0.535684,0.393743,0.690245,0.594286,0.534531,0.370569,0.693004,0.588288
2,TFLite FLOAT16,0.514932,0.381971,0.679067,0.582926,0.513424,0.360064,0.692673,0.566982
3,TFLite INT8,0.516634,0.383540,0.701030,0.568108,0.517406,0.361134,0.701854,0.566605



Total Test Images: 202

Benchmark image:
/kaggle/working/dataset/test/images/ppd_000063_jpg.rf.A7pBk90G99WZFysZUOoS.jpg


BENCHMARKING: PyTorch (.pt)
Running warmup...
Running benchmark...
Model: PyTorch (.pt)
avg_ms: 600.8370
fps: 1.6643
latency_std: 155.4493
size_mb: 22.2606
params_M: 10.3688


BENCHMARKING: TFLite FLOAT32
Running warmup...
Loading /kaggle/working/model_saved_model/model_float32.tflite for TensorFlow Lite inference...
Running benchmark...
Model: TFLite FLOAT32
avg_ms: 671.8706
fps: 1.4884
latency_std: 8.4205
size_mb: 40.0565
params_M: 10.3688


BENCHMARKING: TFLite FLOAT16
Running warmup...
Loading /kaggle/working/model_saved_model/model_float16.tflite for TensorFlow Lite inference...
Running benchmark...
Model: TFLite FLOAT16
avg_ms: 676.5909
fps: 1.4780
latency_std: 7.4604
size_mb: 20.2096
params_M: 10.3688


BENCHMARKING: TFLite INT8
Running warmup...
Loading /kaggle/working/model_saved_model/model_int8.tflite for TensorFlow Lite inference...
Running benchmark...

,Model,avg_ms,fps,latency_std,size_mb,params_M,compression_ratio_vs_pt,speedup_vs_pt
0,PyTorch (.pt),600.837014,1.664345,155.449289,22.260587,10.368823,1.000000,1.000000
1,TFLite FLOAT32,671.870620,1.488382,8.420461,40.056527,10.368823,1.799437,0.894275
2,TFLite FLOAT16,676.590935,1.477998,7.460442,20.209584,10.368823,0.907864,0.888036
3,TFLite INT8,771.284872,1.296538,10.380316,10.652570,10.368823,0.478539,0.779008
